### In order to run the bellow cells, download Amazon datasets for electronics from https://amazon-reviews-2023.github.io/main.html and place them in the /data folder.

In [ ]:
import json
import pandas as pd


In [ ]:
with open('../../data/meta_Electronics.jsonl', 'r') as f:
    first_line = json.loads(f.readline())
    
first_line


In [ ]:
def filter_item(data: dict) -> bool:
    if int(data['details']['Date First Available'][-4:]) < 2022:
        return True
    
    return False

### Select items which are avaialble after 2022, so filter out items are older than before 2022.

In [ ]:
with open('../../data/meta_Electronics.jsonl', 'r') as f:
    with open('../../data/meta_Electronics_2022_onwards.jsonl', 'a', encoding='utf-8') as f_out:
        with open('../../data/meta_Electronics_2022_onwards_no_date.jsonl', 'a', encoding='utf-8') as f_out_no_date:
            line_count = 0
            for line in f:
                data = json.loads(line.strip())
                try:
                    if not filter_item(data):
                        json.dump(data, f_out)
                        f_out.write('\n')
                        f_out.flush()
                except Exception as e:
                    json.dump(data, f_out_no_date)
                    f_out_no_date.write('\n')
                    f_out_no_date.flush()
                
                line_count += 1
                if line_count % 10000 == 0:
                    print(f"Processed {line_count} lines")
                



### Split the items into "Main Category" or Doesn't have any category

In [ ]:
def filter_category(data: dict) -> bool:
    if data['main_category'] == None or data['main_category'] == '':
        return True
    
    return False



In [ ]:
with open('../../data/meta_Electronics_2022_onwards.jsonl', 'r') as f:
    with open('../../data/meta_Electronics_2022_onwards_with_category.jsonl', 'a', encoding='utf-8') as f_out:
        with open('../../data/meta_Electronics_2022_onwards_no_category.jsonl', 'a', encoding='utf-8') as f_out_no_category:
            line_count = 0
            for line in f:
                data = json.loads(line.strip())
                if filter_category(data):
                    json.dump(data, f_out_no_category)
                    f_out_no_category.write('\n')
                    f_out_no_category.flush()
                else:
                    json.dump(data, f_out)
                    f_out.write('\n')
                    f_out.flush()
                
                line_count += 1
                if line_count % 10000 == 0:
                    print(f"Processed {line_count} lines")


### Explore distribution of item categories.

In [ ]:
df = pd.read_json('../../data/meta_Electronics_2022_onwards_with_category.jsonl', lines=True)
df.head()

In [ ]:
df["main_category"].value_counts().plot(kind="bar")

### Filter out items that have at least 100 ratings

In [ ]:
df_with_ratings_100 = df[df["rating_number"] > 100]
len(df)
len(df_with_ratings_100)

In [ ]:
df_with_ratings_100["main_category"].value_counts().plot(kind="bar")

In [ ]:
df_with_ratings_100["average_rating"].plot(kind="hist", bins=50, range=(0, 5))

In [ ]:
df_sample_1000 = df_with_ratings_100.sample(1000, random_state=20)
df_sample_1000["average_rating"].plot(kind="hist", bins=50, range=(0, 5))


In [ ]:
df_sample_1000['main_category'].value_counts().plot(kind="bar")

In [ ]:
df_sample_1000['price'].plot(kind="hist", bins=100, range=(0, 500))

In [ ]:
df_with_ratings_100.to_json('../../data/meta_Electronics_2022_onwards_with_ratings_100.jsonl', orient='records', lines=True)

In [ ]:
df_sample_1000.to_json('../../data/meta_Electronics_2022_onwards_with_ratings_100_sample_1000.jsonl', orient='records', lines=True)

### Extract ratings that match sampled data.

In [ ]:
df_ratings_100 = pd.read_json('../../data/meta_Electronics_2022_onwards_with_ratings_100.jsonl', lines=True)
df_sample_1000 = pd.read_json('../../data/meta_Electronics_2022_onwards_with_ratings_100_sample_1000.jsonl', lines=True)

In [ ]:
with open('../../data/Electronics.jsonl', 'r') as f:
    with open('../../data/Electronics_2022_onwards_with_ratings_100.jsonl', 'a', encoding='utf-8') as f_out:
        id_list = set(df_ratings_100['parent_asin'].values)
        line_count = 0
        for line in f:
            data = json.loads(line.strip())
            if data['parent_asin'] in id_list:
                json.dump(data, f_out)
                f_out.write('\n')
                f_out.flush()
            line_count += 1
            if line_count % 10000 == 0:
                print(f"Processed {line_count} lines")

In [ ]:
with open('../../data/Electronics_2022_onwards_with_ratings_100.jsonl', 'r') as f:
    with open('../../data/Electronics_2022_onwards_with_ratings_100_sample_1000.jsonl', 'a', encoding='utf-8') as f_out:
        id_list = set(df_sample_1000['parent_asin'].values)
        line_count = 0
        for line in f:
            data = json.loads(line.strip())
            if data['parent_asin'] in id_list:
                json.dump(data, f_out)
                f_out.write('\n')
                f_out.flush()
            line_count += 1
            if line_count % 10000 == 0:
                print(f"Processed {line_count} lines")
